# 05 - Pretrained baselines

**What this notebook does**: trains ImageNet-pretrained backbones with a small classification head on
the `faithful` split, so MiniConvNet can be compared against standard architectures on identical
data.

**Default set** (run these): `ResNet50`, `VGG16`, `MobileNetV3Small`, `EfficientNetV2B0`.
**Extended set** (optional, in a clearly marked cell at the end - run only if time allows):
`VGG19`, `InceptionV3`, `ConvNeXtTiny`.

**What must already exist**: split CSVs from notebook 00, and internet access enabled on Kaggle so
Keras can download the ImageNet weights (Notebook settings -> Internet -> On). Without it,
`weights='imagenet'` fails and you would silently be training from scratch if you changed it to
`None`.

**Protocol**: feature extraction - the backbone is frozen (`trainable_base=False`), only the head is
trained, `Adam(1e-4, clipnorm=1.0)`. Each architecture's own `preprocess_input` is applied *inside*
the model, so the shared `[0, 255]` data pipeline is correct for every model here.

**Where results go**: `outputs/results_table.csv` as canonical rows (one per model), since a single
feature-extraction run is the standard comparison protocol. Every run is still collapse-checked.

**What "looks right"**: baselines reaching clearly above-chance accuracy within a few epochs (frozen
ImageNet features converge fast). A baseline stuck near 0.25 usually means the weights failed to
download or preprocessing was bypassed.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('baseline epochs:', EPOCHS_BASELINE, '| lr:', LR_BASELINE)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import load_split, make_split_datasets, split_counts
from src.models import build_baseline, count_params, DEFAULT_BASELINES, EXTENDED_BASELINES
from src.train_utils import (set_global_seeds, gpu_report, compile_model, optimizer_summary,
                            class_weights_for, make_callbacks, save_history, plot_history,
                            final_epoch_summary, run_name_for)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                               plot_confusion_matrix, tumor_vs_subtype_breakdown,
                               result_row_from_metrics, record_canonical, load_results,
                               save_predictions)

set_global_seeds(SEED)
print(gpu_report())
print('default baselines :', DEFAULT_BASELINES)
print('extended baselines:', EXTENDED_BASELINES)

## 1. Data (faithful split, identical to MiniConvNet's)

**Looks right**: the same counts as notebook 00 - the comparison is only meaningful if every model
sees exactly the same images.

In [ ]:
SPLIT_FOR_BASELINES = 'faithful'

sdf = load_split(SPLIT_FOR_BASELINES)
train_ds, val_ds, test_ds, frames = make_split_datasets(sdf)
class_weight = class_weights_for(SPLIT_FOR_BASELINES, frames['train']['label'].values)

print(split_counts(sdf))
print('class_weight:', class_weight if class_weight else 'None')

## 2. Baseline runner

One function per architecture: build (frozen backbone + head), compile, fit, evaluate,
collapse-check, plot, record.

In [ ]:
def run_baseline(name, epochs=EPOCHS_BASELINE, trainable_base=False):
    set_global_seeds(SEED)
    run_name = run_name_for(name.lower(), None, SPLIT_FOR_BASELINES)
    print('=' * 70)
    print('baseline:', name, '| run:', run_name)

    model = build_baseline(name, trainable_base=trainable_base)
    compile_model(model, lr=LR_BASELINE)
    params = count_params(model)
    print('params   :', params)
    print('optimizer:', optimizer_summary(model))

    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                        class_weight=class_weight,
                        callbacks=make_callbacks(run_name), verbose=2)
    save_history(history, run_name)
    print('\n', final_epoch_summary(history))

    y_true, y_pred, y_prob = predict(model, test_ds)
    metrics = compute_metrics(y_true, y_pred, y_prob)

    # LESSON 11: raw predictions for every run, baselines included.
    save_predictions(run_name, y_true, y_pred, y_prob,
                     meta={'arch_variant': 'transfer_frozen' if not trainable_base
                                           else 'transfer_finetuned',
                           'split_variant': SPLIT_FOR_BASELINES, 'baseline': name})

    # Includes the partial-collapse check (LESSON 11).
    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)
    print('\ntest metrics:', {k: round(v, 4) for k, v in metrics.items()})
    print()
    print_collapse_report(collapse, run_name)

    plot_history(history, run_name)
    plot_confusion_matrix(y_true, y_pred, run_name)
    breakdown = tumor_vs_subtype_breakdown(y_true, y_pred)
    print('tumour-vs-subtype:', {k: (round(v, 4) if isinstance(v, float) else v)
                                 for k, v in breakdown.items()})

    record_canonical(result_row_from_metrics(
        model_name=name, metrics=metrics, collapse=collapse,
        arch_variant='transfer_frozen' if not trainable_base else 'transfer_finetuned',
        split_variant=SPLIT_FOR_BASELINES, params=params['total_params'],
        epochs_trained=final_epoch_summary(history)['epochs_trained'], n_runs=1,
        notes=(f"ImageNet feature extraction, head-only training; "
               f"binary_tumor_acc={breakdown['binary_tumor_vs_healthy_accuracy']:.4f}; "
               f"subtype_acc={breakdown['subtype_accuracy_all_tumors']:.4f}")))

    tf.keras.backend.clear_session()
    return {'model': name, 'run_name': run_name, 'status': collapse['status'],
            'params': params['total_params'], **metrics}

## 3. Default baselines

Each architecture gets its own cell so a failed download or an OOM only costs you that one model.

In [ ]:
results = {}
results['ResNet50'] = run_baseline('ResNet50')

In [ ]:
results['VGG16'] = run_baseline('VGG16')

In [ ]:
results['MobileNetV3Small'] = run_baseline('MobileNetV3Small')

In [ ]:
results['EfficientNetV2B0'] = run_baseline('EfficientNetV2B0')

## 4. Default-set summary

**Looks right**: four rows, all `ok`, all with far more parameters than MiniConvNet - that parameter
gap is the point of the comparison.

In [ ]:
tbl = pd.DataFrame(results.values())
print(tbl[['model', 'params', 'accuracy', 'f1_macro', 'cohen_kappa', 'mcc', 'status']]
      .round(4).to_string(index=False))

collapsed = tbl[tbl['status'] != VALID_TAG]
print('\ncollapsed baselines:', collapsed['model'].tolist() if len(collapsed) else 'none')

## 5. OPTIONAL - extended baselines

**Run only if time allows.** These add `VGG19`, `InceptionV3` and `ConvNeXtTiny`. They roughly double
the runtime of this notebook and none of them is required for the paper comparison. Skipping this
cell leaves every earlier result intact.

Note: `InceptionV3` officially expects 299x299 inputs; it accepts 224x224 with `include_top=False`,
and keeping the shared 224x224 pipeline is the deliberate choice here so that every model sees
identical data. Record that if you report it.

In [ ]:
RUN_EXTENDED = False   # <- set to True only if you have time to spare

if RUN_EXTENDED:
    for name in EXTENDED_BASELINES:
        results[name] = run_baseline(name)
    print('\nextended baselines complete:', EXTENDED_BASELINES)
else:
    print('Skipped the extended baseline set (RUN_EXTENDED = False).')
    print('Set RUN_EXTENDED = True and re-run this cell to add:', EXTENDED_BASELINES)

## 6. Canonical table so far

**Looks right**: one row per baseline plus the two MiniConvNet CV rows from notebook 04.

In [ ]:
canon = load_results('canonical')
print(canon[['model', 'arch_variant', 'split_variant', 'params', 'accuracy',
             'accuracy_std', 'f1_macro', 'n_runs', 'status']].round(4).to_string(index=False))
print('\nnext: 06_evaluate_and_compare.ipynb')